<div style="
  background-color:#CD5C5C;
  padding:20px;
  border-radius:12px;
  text-align:center;
  color:white;
  margin-bottom:20px;
">

  <h1 style="margin:0;">
    Data exploration — Analyse e-commerce
  </h1>

  <p style="margin:5px 0 0 0; font-size:16px;">
    Notebook 01 : Analyse Exploratoire des Données
  </p>

</div>

---

#  ÉTAPE 1 : Analyse exploratoire

In [ ]:
"""
Configuration de l'environnement — Analyse e-commerce
===================================================
Imports et configuration pour l'analyse exploratoire des données de churn.
"""

import os
import sys

# Manipulation des données
import pandas as pd
import numpy as np

# Statistiques et tests
import scipy.stats as ss

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Seed pour la reproductibilité
RANDOM_STATE = 1204

# Import de la fonction utilitaire depuis utils/data_prep.py
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.data_prep import detect_possible_outliers

print(f" Imports chargés avec succès")

 Imports chargés avec succès


In [3]:
# Vérification du répertoire de travail
print(f" Répertoire courant : {os.getcwd()}")

 Répertoire courant : c:\Users\juber\Documents\Analyse-e-commerce-GCP-BigQuery\notebooks


In [4]:
# ============================================================
# Chargement du dataset brut
# ============================================================
df = pd.read_csv("..\\data\\thelook_fr_women_2023_2024.csv")
print(f" Dataset importé : {df.shape[0]} lignes et {df.shape[1]} colonnes\n")

 Dataset importé : 1679 lignes et 20 colonnes



In [ ]:
# Aperçu des premières lignes aléatoires du dataset
df.sample(10)

,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city
1022,19424,28111,10459,2024-05-31 08:51:21+00:00,Processing,29.000000,15.428000,Intimates,Women,Wonderbra,Wonderbra Women's Statement Makers Adjustable ...,Processing,2024-05-31 10:00:00+00:00,NaN,NaN,15644,F,France,Centre-Val de Loire,Corquilleroy
853,103135,149809,14118,2024-03-09 04:45:34+00:00,Shipped,3.700000,1.346800,Accessories,Women,Locs,LOCS Sunglasses Hardcore Black Dark Lens 0103 ...,Shipped,2024-03-05 06:05:00+00:00,2024-03-07 18:37:00+00:00,NaN,82405,F,France,Île-de-France,Évry-Courcouronnes
634,18732,27096,489,2023-12-07 09:03:33+00:00,Returned,39.500000,21.330000,Tops & Tees,Women,Volcom,Volcom Juniors Moclov Crew Top,Returned,2023-12-07 11:09:00+00:00,2023-12-08 05:47:00+00:00,2023-12-12 14:18:00+00:00,15097,F,France,Hauts-de-France,Lille
995,91980,133674,8854,2024-05-17 03:28:56+00:00,Complete,10.990000,4.033330,Socks & Hosiery,Women,Foot Traffic,Soft and Warm Microfiber Fuzzy Socks in Your C...,Complete,2024-05-17 04:08:00+00:00,2024-05-17 18:23:00+00:00,2024-05-20 22:43:00+00:00,73415,F,France,Centre-Val de Loire,Ardon
1479,70547,102453,8734,2024-11-02 06:30:12+00:00,Returned,158.330002,65.073631,Outerwear & Coats,Women,Tommy Hilfiger,Tommy Hilfiger Women's Leather Jacket,Returned,2024-11-02 06:43:00+00:00,2024-11-02 12:37:00+00:00,2024-11-02 13:50:00+00:00,56441,F,France,Île-de-France,Argenteuil
1060,99261,144190,6147,2024-06-12 11:22:28+00:00,Processing,20.000000,11.560000,Leggings,Women,Torrid,Torrid Plus Size Black Lace Capri Leggings,Processing,2024-06-12 13:11:00+00:00,NaN,NaN,79356,F,France,Nouvelle-Aquitaine,Boulazac Isle Manoire
60,1079,1554,5925,2023-02-08 02:40:50+00:00,Processing,10.530000,6.065280,Leggings,Women,Allegra K,Allegra K Ladies Elastic Waist Cartoon Cats Pa...,Processing,2023-02-05 02:25:00+00:00,NaN,NaN,883,F,France,Hauts-de-France,Bulles
982,114147,165801,2978,2024-05-11 15:38:33+00:00,Shipped,77.000000,28.644000,Active,Women,Beyond Yoga,Beyond Yoga Women's Original Pant,Shipped,2024-05-07 17:50:00+00:00,2024-05-09 07:31:00+00:00,NaN,91268,F,France,Auvergne-Rhône-Alpes,Anse
1279,89232,129695,9417,2024-08-24 12:30:01+00:00,Complete,11.990000,5.371520,Socks & Hosiery,Women,dollhouse,Dollhouse Women's 2-Pairs Thick Boot Socks - M...,Complete,2024-08-24 12:38:00+00:00,2024-08-26 19:01:00+00:00,2024-08-29 07:20:00+00:00,71238,F,France,Île-de-France,Vaux-le-Pénil
1392,62851,91269,15446,2024-10-02 21:44:15+00:00,Cancelled,14.950000,7.774000,Plus,Women,Knit Caps,Cable Knit Winter Ski Beret Knit Tam Skull Hat...,Cancelled,2024-10-03 01:08:00+00:00,NaN,NaN,50331,F,France,Bretagne,Brest


In [6]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          1679 non-null   int64  
 1   order_item_id     1679 non-null   int64  
 2   product_id        1679 non-null   int64  
 3   item_created_at   1679 non-null   str    
 4   item_status       1679 non-null   str    
 5   sale_price        1679 non-null   float64
 6   cost              1679 non-null   float64
 7   category          1679 non-null   str    
 8   department        1679 non-null   str    
 9   brand             1677 non-null   str    
 10  product_name      1679 non-null   str    
 11  order_status      1679 non-null   str    
 12  order_created_at  1679 non-null   str    
 13  shipped_at        1133 non-null   str    
 14  delivered_at      636 non-null    str    
 15  user_id           1679 non-null   int64  
 16  gender            1679 non-null   str    
 17  countr

In [ ]:
# Description des colonnes
df.describe(include='all') 

,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city
count,1679.000000,1679.000000,1679.000000,1679,1679,1679.000000,1679.000000,1679,1679,1677,1679,1679,1679,1133,636,1679.000000,1679,1679,1679,1679
unique,NaN,NaN,NaN,1679,5,NaN,NaN,22,1,657,1559,5,1117,747,416,NaN,1,1,13,547
top,NaN,NaN,NaN,2023-01-01 06:18:03+00:00,Shipped,NaN,NaN,Intimates,Women,Allegra K,Suit Studio Garden Grove Skirt Suit,Shipped,2023-01-14 11:28:00+00:00,2023-01-16 08:38:00+00:00,2023-04-07 04:06:00+00:00,NaN,F,France,Île-de-France,Paris
freq,NaN,NaN,NaN,1,497,NaN,NaN,247,1679,101,3,497,4,4,4,NaN,1679,1679,397,94
mean,60851.683740,88372.698035,7922.506254,NaN,NaN,57.021769,27.460066,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48702.836212,NaN,NaN,NaN,NaN
std,35655.502994,51820.578196,4680.553838,NaN,NaN,69.682245,31.854450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28441.847629,NaN,NaN,NaN,NaN
min,359.000000,517.000000,12.000000,NaN,NaN,1.820000,0.749840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,279.000000,NaN,NaN,NaN,NaN
25%,30357.000000,44069.500000,3812.000000,NaN,NaN,19.990000,9.681000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24349.000000,NaN,NaN,NaN,NaN
50%,60827.000000,88353.000000,7907.000000,NaN,NaN,38.000000,18.230800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48715.000000,NaN,NaN,NaN,NaN
75%,90911.000000,132106.000000,12053.000000,NaN,NaN,68.000000,33.232345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72541.000000,NaN,NaN,NaN,NaN


In [10]:
# Description des colonnes numériques
df.describe(include=[np.number])

,order_id,order_item_id,product_id,sale_price,cost,user_id
count,1679.000000,1679.000000,1679.000000,1679.000000,1679.000000,1679.000000
mean,60851.683740,88372.698035,7922.506254,57.021769,27.460066,48702.836212
std,35655.502994,51820.578196,4680.553838,69.682245,31.854450,28441.847629
min,359.000000,517.000000,12.000000,1.820000,0.749840,279.000000
25%,30357.000000,44069.500000,3812.000000,19.990000,9.681000,24349.000000
50%,60827.000000,88353.000000,7907.000000,38.000000,18.230800,48715.000000
75%,90911.000000,132106.000000,12053.000000,68.000000,33.232345,72541.000000
max,124791.000000,181230.000000,15982.000000,903.000000,437.052001,99717.000000


In [11]:
# Description des colonnes catégorielles
df.describe(include=['object'])

C:\Users\juber\AppData\Local\Temp\ipykernel_19100\2147888001.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=['object'])


,item_created_at,item_status,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,gender,country,state,city
count,1679,1679,1679,1679,1677,1679,1679,1679,1133,636,1679,1679,1679,1679
unique,1679,5,22,1,657,1559,5,1117,747,416,1,1,13,547
top,2023-01-01 06:18:03+00:00,Shipped,Intimates,Women,Allegra K,Suit Studio Garden Grove Skirt Suit,Shipped,2023-01-14 11:28:00+00:00,2023-01-16 08:38:00+00:00,2023-04-07 04:06:00+00:00,F,France,Île-de-France,Paris
freq,1,497,247,1679,101,3,497,4,4,4,1679,1679,397,94


In [21]:
col_list = df.columns.tolist()
print(f" Colonnes du dataset : {len(col_list)} colonnes\n{col_list}")

 Colonnes du dataset : 20 colonnes
['order_id', 'order_item_id', 'product_id', 'item_created_at', 'item_status', 'sale_price', 'cost', 'category', 'department', 'brand', 'product_name', 'order_status', 'order_created_at', 'shipped_at', 'delivered_at', 'user_id', 'gender', 'country', 'state', 'city']


In [12]:
# Vérification des doublons sur le jeu d'entraînement
print("\n" + "=" * 70)
print("  VÉRIFICATION DES DOUBLONS ")
print("=" * 70 + "\n")

# Doublons exacts
n_doublons_exacts = df.duplicated().sum()
pct_doublons_exacts = (n_doublons_exacts / len(df)) * 100
print(f" Doublons exacts (toutes colonnes)           : {n_doublons_exacts} ({pct_doublons_exacts:.2f}%)")

# Doublons sans tenir compte de customerID (supposé unique)
if "customerID" in df.columns:
    n_doublons_no_id = df.duplicated(subset=df.columns.difference(["customerID"])).sum()
    pct_doublons_no_id = (n_doublons_no_id / len(df)) * 100
    print(f" Doublons (hors customerID)                  : {n_doublons_no_id} ({pct_doublons_no_id:.2f}%)")

print("\n" + "=" * 70)


  VÉRIFICATION DES DOUBLONS 

 Doublons exacts (toutes colonnes)           : 0 (0.00%)



In [32]:
df_data = df.select_dtypes(include = ["number"], exclude = ["float64"])
df_data 

,order_id,order_item_id,product_id,user_id
0,19425,28112,6983,15644
1,19425,28113,10597,15644
2,4710,6730,11792,3853
3,16618,24006,329,13422
4,4710,6731,5295,3853
...,...,...,...,...
1674,8816,12714,12818,7185
1675,121884,177090,12995,97447
1676,97667,141881,6393,78033
1677,84954,123434,653,67863


In [33]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          1679 non-null   int64  
 1   order_item_id     1679 non-null   int64  
 2   product_id        1679 non-null   int64  
 3   item_created_at   1679 non-null   str    
 4   item_status       1679 non-null   str    
 5   sale_price        1679 non-null   float64
 6   cost              1679 non-null   float64
 7   category          1679 non-null   str    
 8   department        1679 non-null   str    
 9   brand             1677 non-null   str    
 10  product_name      1679 non-null   str    
 11  order_status      1679 non-null   str    
 12  order_created_at  1679 non-null   str    
 13  shipped_at        1133 non-null   str    
 14  delivered_at      636 non-null    str    
 15  user_id           1679 non-null   int64  
 16  gender            1679 non-null   str    
 17  countr

In [35]:
cols = ["item_created_at", "order_created_at", "shipped_at", "delivered_at"]

for col in cols:
    df[col] = pd.to_datetime(df[col])

In [36]:
#Vérification des types de données après conversion
print("\n Types de données après conversion :")
print(df.dtypes)


 Types de données après conversion :
order_id                          int64
order_item_id                     int64
product_id                        int64
item_created_at     datetime64[us, UTC]
item_status                         str
sale_price                      float64
cost                            float64
category                            str
department                          str
brand                               str
product_name                        str
order_status                        str
order_created_at    datetime64[us, UTC]
shipped_at          datetime64[us, UTC]
delivered_at        datetime64[us, UTC]
user_id                           int64
gender                              str
country                             str
state                               str
city                                str
dtype: object


In [45]:
# Analyse des valeurs manquantes
print("\n" + "=" * 70)
print("  ANALYSE DES VALEURS MANQUANTES")
print("=" * 70 + "\n")

missing_data = pd.DataFrame({
    "Colonne": df.columns,
    "Manquantes": df.isnull().sum().values,
    "Pourcentage": (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_data_filtered = missing_data[missing_data["Manquantes"] > 0].sort_values(by="Manquantes", ascending=False)

if len(missing_data_filtered) > 0:
    print("  Colonnes avec valeurs manquantes :\n")
    display(missing_data_filtered.reset_index(drop=True))
else:
    print(" Aucune valeur manquante détectée dans df !")

print(f"\n Total de cellules manquantes     : {df.isnull().sum().sum()}")
print(f" Pourcentage global de complétude : {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")
print("\n" + "=" * 70)


  ANALYSE DES VALEURS MANQUANTES

  Colonnes avec valeurs manquantes :



,Colonne,Manquantes,Pourcentage
0,delivered_at,1043,62.12
1,shipped_at,546,32.52
2,brand,2,0.12



 Total de cellules manquantes     : 1591
 Pourcentage global de complétude : 95.26%



In [46]:
# Types de données et complétude
print("\n" + "=" * 70)
print("  INFORMATIONS SUR LES COLONNES & TYPES DE DONNÉES")
print("=" * 70 + "\n")

info_df = pd.DataFrame({
    "Colonne": df.columns,
    "Type": df.dtypes.values,
    "Non-Null": df.notnull().sum().values,
    "Null": df.isnull().sum().values,
    "Null %": (df.isnull().sum().values / len(df) * 100).round(2)
})
display(info_df)

print("\n RÉSUMÉ DES TYPES :")
print(f"   • Variables numériques (int/float) : {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"   • Variables catégorielles (object)  : {df.select_dtypes(include=['object']).shape[1]}")
print(f"   • Complétude globale              : {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")


  INFORMATIONS SUR LES COLONNES & TYPES DE DONNÉES



,Colonne,Type,Non-Null,Null,Null %
0,order_id,int64,1679,0,0.00
1,order_item_id,int64,1679,0,0.00
2,product_id,int64,1679,0,0.00
3,item_created_at,"datetime64[us, UTC]",1679,0,0.00
4,item_status,str,1679,0,0.00
5,sale_price,float64,1679,0,0.00
6,cost,float64,1679,0,0.00
7,category,str,1679,0,0.00
8,department,str,1679,0,0.00
9,brand,str,1677,2,0.12



 RÉSUMÉ DES TYPES :
   • Variables numériques (int/float) : 6
   • Variables catégorielles (object)  : 10
   • Complétude globale              : 95.26%


C:\Users\juber\AppData\Local\Temp\ipykernel_19100\399955216.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(f"   • Variables catégorielles (object)  : {df.select_dtypes(include=['object']).shape[1]}")


In [50]:
fig = px.box(
    df,
    x="category",
    y="sale_price",
    color="category"
)

fig.show()

In [51]:
df["margin"] = df["sale_price"] - df["cost"]

In [52]:
fig = px.box(
    df,
    x="department",
    y="margin",
    color="department"
)

fig.show()

In [54]:
df["delivery_time"] = (
    df["delivered_at"] - df["shipped_at"]
).dt.days

In [57]:
num_cols = ["sale_price", "cost"]

fig = make_subplots(
    rows=1,
    cols=len(num_cols),
    subplot_titles=[
        "Distribution du prix de vente (€)",
        "Distribution du coût (€)"
    ]
)

for i, col in enumerate(num_cols, 1):
    fig.add_trace(
        go.Box(
            y=df[col],
            name=col,
            marker_color="#011c5d",
            boxmean=True,
            boxpoints="outliers"
        ),
        row=1, col=i
    )

fig.update_layout(
    title={
        "text": "📦 Analyse des variables numériques",
        "x": 0.5,
        "xanchor": "center"
    },
    height=500,
    width=900,
    showlegend=False
)

# Titres des axes
fig.update_yaxes(title_text="Montant (€)", row=1, col=1)
fig.update_yaxes(title_text="Montant (€)", row=1, col=2)

fig.show()

In [58]:
# Analyse détaillée des valeurs manquantes et recommandations
print("\n" + "=" * 70)
print("  STRATÉGIE DE TRAITEMENT DES VALEURS MANQUANTES")
print("=" * 70 + "\n")

missing_analysis = pd.DataFrame({
    "Colonne": df.columns,
    "Manquants": df.isnull().sum().values,
    "Manquants %": (df.isnull().sum().values / len(df) * 100).round(2),
    "Status": [" OK" if x == 0 else "  À traiter" for x in df.isnull().sum().values]
})

missing_analysis = missing_analysis[missing_analysis["Manquants"] > 0].sort_values(by="Manquants", ascending=False)

if len(missing_analysis) == 0:
    print(" EXCELLENT : Aucune valeur manquante détectée dans X_train !")
    print("   Le dataset est complet et prêt pour la modélisation.")
    print("\n   Recall : TotalCharges a été nettoyé lors du chargement")
    print("   (les espaces vides ont été convertis et les lignes supprimées).")
else:
    print("  VALEURS MANQUANTES DÉTECTÉES :\n")
    display(missing_analysis)
    print("\n RECOMMANDATIONS :")
    for _, row in missing_analysis.iterrows():
        pct = row["Manquants %"]
        if pct < 1:
            print(f"   • {row['Colonne']} ({pct}%) : Supprimer les lignes")
        elif pct < 5:
            print(f"   • {row['Colonne']} ({pct}%) : Imputer (moyenne/médiane/mode)")
        elif pct < 30:
            print(f"   • {row['Colonne']} ({pct}%) : Analyser le pattern d'absence")
        else:
            print(f"   • {row['Colonne']} ({pct}%) :  Considérer la suppression")


  STRATÉGIE DE TRAITEMENT DES VALEURS MANQUANTES

  VALEURS MANQUANTES DÉTECTÉES :



,Colonne,Manquants,Manquants %,Status
14,delivered_at,1043,62.12,À traiter
21,delivery_time,1043,62.12,À traiter
13,shipped_at,546,32.52,À traiter
9,brand,2,0.12,À traiter



 RECOMMANDATIONS :
   • delivered_at (62.12%) :  Considérer la suppression
   • delivery_time (62.12%) :  Considérer la suppression
   • shipped_at (32.52%) :  Considérer la suppression
   • brand (0.12%) : Supprimer les lignes


In [60]:
# Histogramme des prix de vente
# Objectif : analyser la distribution, détecter les valeurs extrêmes
fig = px.histogram(
    df,
    x="sale_price",
    nbins=50,
    title="Distribution des prix de vente (€)"
)

fig.update_layout(
    xaxis_title="Prix de vente (€)",
    yaxis_title="Fréquence"
)

fig.show()


# Histogramme des coûts
# Objectif : comparer la structure des coûts avec les prix
fig = px.histogram(
    df,
    x="cost",
    nbins=50,
    title="Distribution des coûts (€)"
)

fig.update_layout(
    xaxis_title="Coût (€)",
    yaxis_title="Fréquence"
)

fig.show()

In [61]:
# Chiffre d'affaires par marque
# Objectif : identifier les marques les plus performantes
ca_brand = (
    df
    .groupby("brand")["sale_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig = px.bar(
    ca_brand,
    title="Top 10 des marques par chiffre d’affaires"
)

fig.update_layout(
    xaxis_title="Marque",
    yaxis_title="Chiffre d’affaires (€)"
)

fig.show()


# Chiffre d'affaires par catégorie
# Objectif : analyser la contribution des catégories
ca_category = (
    df
    .groupby("category")["sale_price"]
    .sum()
    .sort_values(ascending=False)
)

px.bar(
    ca_category,
    title="Chiffre d’affaires par catégorie",
    labels={"value": "CA (€)", "category": "Catégorie"}
).show()


# Chiffre d'affaires par ville
# Objectif : analyser la performance géographique
ca_city = (
    df
    .groupby("city")["sale_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

px.bar(
    ca_city,
    title="Top 10 des villes par chiffre d’affaires",
    labels={"value": "CA (€)", "city": "Ville"}
).show()

In [62]:
# Conversion des dates
# Objectif : permettre les analyses temporelles
df["order_created_at"] = pd.to_datetime(df["order_created_at"])

# Création de variables temporelles
df["year"] = df["order_created_at"].dt.year
df["month"] = df["order_created_at"].dt.to_period("M")

# Calcul du chiffre d'affaires mensuel
monthly_sales = (
    df
    .groupby("month")["sale_price"]
    .sum()
    .reset_index()
)

monthly_sales["month"] = monthly_sales["month"].astype(str)

# Visualisation de la tendance
fig = px.line(
    monthly_sales,
    x="month",
    y="sale_price",
    title="Évolution mensuelle du chiffre d’affaires",
    markers=True
)

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="Chiffre d’affaires (€)"
)

fig.show()

C:\Users\juber\AppData\Local\Temp\ipykernel_19100\3702100402.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["order_created_at"].dt.to_period("M")


In [63]:
# Agrégation par année et mois
monthly = (
    df
    .groupby(["year", "month"])["sale_price"]
    .sum()
    .reset_index()
)

monthly["month_str"] = monthly["month"].astype(str)

# Visualisation comparative
fig = px.line(
    monthly,
    x="month_str",
    y="sale_price",
    color="year",
    title="Comparaison mensuelle 2023 vs 2024"
)

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="Chiffre d’affaires (€)"
)

fig.show()

In [64]:
# Création d'un tableau pivot
# Objectif : comparer directement les deux années
pivot = monthly.pivot(
    index="month",
    columns="year",
    values="sale_price"
)

# Calcul de l'écart absolu
pivot["diff_abs"] = pivot[2024] - pivot[2023]

# Calcul de l'écart relatif (%)
pivot["diff_pct"] = (pivot["diff_abs"] / pivot[2023]) * 100

pivot = pivot.reset_index()
pivot["month"] = pivot["month"].astype(str)

In [67]:
df["year"].value_counts()

year
2024    968
2023    709
2022      2
Name: count, dtype: int64

In [68]:
df_filtered = df[df["year"].isin([2023, 2024])]

In [69]:
df_filtered["month_num"] = df_filtered["order_created_at"].dt.month

In [70]:
monthly = (
    df_filtered
    .groupby(["year", "month_num"])["sale_price"]
    .sum()
    .reset_index()
)

In [71]:
pivot = monthly.pivot(
    index="month_num",
    columns="year",
    values="sale_price"
)

In [72]:
pivot = pivot.dropna()

In [73]:
# Écart absolu
pivot["diff_abs"] = pivot[2024] - pivot[2023]

# Écart relatif (%)
pivot["diff_pct"] = (pivot["diff_abs"] / pivot[2023]) * 100

pivot = pivot.reset_index()

In [74]:
# Écart absolu
px.bar(
    pivot,
    x="month_num",
    y="diff_abs",
    title="Écart absolu de CA (2024 vs 2023)",
    labels={"month_num": "Mois", "diff_abs": "Écart (€)"}
).show()


# Écart relatif
px.bar(
    pivot,
    x="month_num",
    y="diff_pct",
    title="Écart relatif (%) (2024 vs 2023)",
    labels={"month_num": "Mois", "diff_pct": "Variation (%)"}
).show()

In [75]:
# Seuil plus réaliste
pivot["anomaly"] = abs(pivot["diff_pct"]) > 15

pivot[pivot["anomaly"]]

year,month_num,2023,2024,diff_abs,diff_pct,anomaly
0,1,2315.510011,3810.400001,1494.889991,64.559859,True
1,2,2621.729992,3647.640007,1025.910016,39.131033,True
3,4,4050.459992,3425.950023,-624.509970,-15.418248,True
4,5,3422.949999,4148.710011,725.760012,21.202764,True
5,6,3833.690012,5503.710015,1670.020003,43.561686,True
6,7,1959.970003,3969.860006,2009.890003,102.546978,True
7,8,3402.479999,4742.120005,1339.640007,39.372458,True
8,9,4446.080003,5452.640004,1006.560002,22.639269,True
9,10,5216.539997,6607.170018,1390.630021,26.658092,True
10,11,2871.210017,5737.620013,2866.409996,99.832822,True


Afin de comparer correctement les performances entre 2023 et 2024, les données ont été agrégées par mois en utilisant uniquement le numéro du mois. Les mois incomplets ont été exclus afin d’assurer une comparaison cohérente.

Cette approche permet d’éviter les biais liés à des périodes non comparables et garantit la pertinence des écarts calculés.

---

#  ÉTAPE 2 : KPI 

Les lignes considérées comme vente : order_status == "Complete"

Les lignes retournées : order_status == "Returned"

Les lignes « vente » excluent tout autre statut (ex: Pending, Cancelled…)

Panier moyen : chiffre d’affaires divisé par le nombre de commandes complètes (sale_price > 0)

Taux de ré-achat : clients ayant ≥2 commandes complètes dans la même année

In [76]:
# Définition des lignes de vente
sales = df[df["order_status"] == "Complete"]

# Vérification
print(f"Lignes de vente : {len(sales)}")

Lignes de vente : 425


In [77]:
# Somme des prix de vente des lignes considérées comme ventes
total_ca = sales["sale_price"].sum()

print(f"Chiffre d'affaires total : {total_ca:.2f} €")

Chiffre d'affaires total : 23522.60 €


In [78]:
# Marge brute = CA - coût
gross_margin = (sales["sale_price"] - sales["cost"]).sum()

print(f"Marge brute totale : {gross_margin:.2f} €")

Marge brute totale : 12209.95 €


In [79]:
# Nombre de commandes avec revenu > 0
# On considère 'order_id' pour compter les commandes
orders_with_revenue = sales[sales["sale_price"] > 0]["order_id"].nunique()

# Panier moyen
average_basket = total_ca / orders_with_revenue

print(f"Panier moyen par commande : {average_basket:.2f} €")

Panier moyen par commande : 83.71 €


In [80]:
# Lignes retournées
returns = df[df["order_status"] == "Returned"]

# Lignes vendues + retours
sold_plus_returns = df[df["order_status"].isin(["Complete", "Returned"])]

# Calcul du taux de retour
return_rate = len(returns) / len(sold_plus_returns) * 100

print(f"Taux de retour : {return_rate:.2f} %")

Taux de retour : 33.18 %


In [81]:
# Extraction de l'année si pas déjà fait
df["order_created_at"] = pd.to_datetime(df["order_created_at"])
sales["year"] = sales["order_created_at"].dt.year

# Nombre de commandes par client et par année
orders_per_client = (
    sales.groupby(["user_id", "year"])["order_id"]
    .nunique()
    .reset_index(name="nb_orders")
)

# Clients avec ≥2 commandes complètes
repeat_customers = orders_per_client[orders_per_client["nb_orders"] >= 2]

# Nombre total de clients uniques
total_clients = sales["user_id"].nunique()

# Taux de ré-achat
repeat_rate = len(repeat_customers["user_id"].unique()) / total_clients * 100

print(f"Taux de ré-achat : {repeat_rate:.2f} %")

Taux de ré-achat : 2.26 %


In [82]:
kpi_summary = {
    "Chiffre d'affaires (€)": total_ca,
    "Marge brute (€)": gross_margin,
    "Panier moyen (€)": average_basket,
    "Taux de retour (%)": return_rate,
    "Taux de ré-achat (%)": repeat_rate
}

import pandas as pd
pd.DataFrame([kpi_summary])

,Chiffre d'affaires (€),Marge brute (€),Panier moyen (€),Taux de retour (%),Taux de ré-achat (%)
0,23522.600014,12209.950302,83.71032,33.176101,2.255639
